In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from openmmtools.integrators import LangevinIntegrator, LangevinSplittingGirsanov
import matplotlib

import torch
from torch import nn, optim, autograd
from torch.nn import functional as F
from torch.utils.data.dataset import random_split
from openmmtorch import TorchForce

import numpy as np
from matplotlib import pyplot as plt

import gc

from scipy.interpolate import CubicSpline
from scipy.spatial import cKDTree

from pymbar import MBAR

from potential import *
from fmrc import *

In [ ]:
##############################################################################
# Global parameters
##############################################################################
mass = 1.0 * dalton
temp = 298.15
temperature = temp * kelvin
collision_rate = 10 / picosecond
timestep = 5 * femtosecond
splitting = 'R V O V R'
nstxout = 20
kt = temp*8.314/1000
platform = Platform.getPlatformByName('CUDA')

#### 0. Unbiased Simulation as Reference

##### 0.1 Mueller Brown Potential

In [ ]:
# Simulation setup
n_sim = 10
n_iters = 1000000

# Run 2D Muller Brown simulation
system = System()
systemforce = MullerForce()
starting_positions = np.concatenate(
    [np.tile([-0.57,1.43,0],(int(n_sim/2),1)),
     np.tile([0.6,0.05,0],(int(n_sim/2),1)),],axis=0
)

# For unbiased simulation, we can parallelize the independent runs.
for n in range(n_sim):
    system.addParticle(mass)
    systemforce.addParticle(n, [])
system.addForce(systemforce)

topology = app.Topology()

# Langevin integrator that allows girsanov reweighting factor output
integrator = LangevinSplittingGirsanov(
    nstxout = nstxout, temperature = temperature,collision_rate = collision_rate,
    timestep = timestep, splitting = splitting
)

# Create simulation
simulation = Simulation(topology,system,integrator,platform)
simulation.context.setPositions(starting_positions)
simulation.context.setVelocitiesToTemperature(temperature)

In [ ]:
%%time
save_dir = 'traj_and_dat/2d_muller/unbiased_{n_sim}x{n_iters}.npy'.format(
    n_sim = n_sim, n_iters = n_iters
)

# Notice we are now running 2D simulations
data = np.zeros((n_sim,n_iters,2))
for j in range(n_iters):
    if j % 1000 == 0:
        print(str(j)+'/'+str(n_iters))
    data[:,j,:] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[:,:2]
    simulation.step(nstxout)

np.save(save_dir,data)

#### 1. FMRC Dimensionality Reduction
To make discretization and visualization easier, we reduce the dimension of the system from 2 to 1 using the FMRC algorithm with the unbiased simulation as training data. We also postulate that the deposition of bias on a 1D collective variable will have a milder effect on the path ensemble, so that Girsanov reweighting will perform better.

In [ ]:
# Read unbiased simulation data
# Using only the first trajectory and stride the trajectory for faster training
data_ref = np.load('traj_and_dat/2d_muller/unbiased_10x1000000.npy')[0]
data_ref.shape

In [ ]:
# TICA pre-processing
dim = None
var_cutoff = None
koopman = False
tica_lagtime = 1    # This is 1ps

# TICA pre-processing
tica,tica_output,tica_output_concat = run_TICA(data_ref,tica_lagtime,dim,var_cutoff,koopman)
data_input = [tica_output]
input_size = tica_output.shape[1]

In [ ]:
### FMRC parameters
# FMRC training
fmrc_lagtime = tica_lagtime
latent_size = 1
hidden_size = 16
hidden_depth = 3
activation = nn.ReLU()
sigma = 0.001
learning_rate = 0.001
lr_decay = 0.1
lr_decay_stepsize = 50
val_frac = 0.1
batch_size = 512
n_epochs = 100
device = 'cuda'

In [ ]:
%%time
# FMRC
fmrc = FMRC(input_size,latent_size,hidden_size,hidden_depth,activation,sigma,learning_rate,lr_decay,
                lr_decay_stepsize,val_frac,batch_size,n_epochs,device)
print(fmrc)
fmrc.fit(data_input,fmrc_lagtime)
fmrc.save_model('models/muller_2d.pt')

In [ ]:
fmrc = torch.load('models/muller_2d.pt')

In [ ]:
# Transform into FMRC space and normalize
r = fmrc.transform(tica_output)

# Visualize
markersize = 3
stride = 1

fig,ax = plt.subplots()


sc = ax.scatter(data_ref[:,0][::stride],data_ref[:,1][::stride],c=r[::stride],s=markersize)
plt.colorbar(sc,label='RC')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim([MullerForce().x_range[0],MullerForce().x_range[1]])
ax.set_ylim([MullerForce().y_range[0],MullerForce().y_range[1]])

# Visualize the whole sample space
x = np.linspace(MullerForce().x_range[0],MullerForce().x_range[1],100)
y = np.linspace(MullerForce().y_range[0],MullerForce().y_range[1],100)
X, Y = np.meshgrid(x,y)
# Flatten grid into (n_points, 2)
XY = np.column_stack([X.ravel(), Y.ravel()])
r_grid = fmrc.transform(tica.transform(XY)).reshape(X.shape)

contour = ax.contour(X, Y, r_grid, levels=50, colors='black',linestyles='--')
#plt.colorbar(contour, label='RC')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
# Histogram with density normalization
counts, bin_edges = np.histogram(r, bins=50, density=True)

# Bin centers
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Avoid log(0) by replacing zeros with a very small number
counts = np.where(counts > 0, counts, 1e-10)

# Compute -log(pdf estimate)
neg_logpdf = -kt * np.log(counts)
neg_logpdf = neg_logpdf - neg_logpdf.min()

# Plot
plt.figure(figsize=(6,4))
plt.plot(bin_centers, neg_logpdf, marker='o', linestyle='-', color='black')
plt.xlabel("Value")
plt.ylabel("-log(pdf estimate)")
plt.title("Negative log of histogram PDF estimate")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
class FMRC_TorchForce(torch.nn.Module):
    def __init__(self,fmrc,tica_evecs,tica_mean):
        super().__init__()
        self.tica_evecs = torch.tensor(tica_evecs,device=device,dtype=torch.float64)
        self.tica_mean = torch.tensor(tica_mean,device=device,dtype=torch.float64)
        self.fmrc = fmrc
    def forward(self, positions):
        # Remove z position, notice that with TorchForce, we cannot run many particles on Muller-Brown in parallel
        positions = positions[:,:2]
        # TICA preprocessing transformation, remove mean first
        tica_output = torch.mm((positions - self.tica_mean),self.tica_evecs)
        # Encode with FMRC
        r = self.fmrc.encode(tica_output)[0][0]      # Only return CV value of the first particle
        return r

# Access TICA mean and eigenvectors
tica_evecs = tica.singular_vectors_left
tica_mean = tica.mean_0

# Wrap into torch jit script
module = torch.jit.script(FMRC_TorchForce(fmrc,tica_evecs,tica_mean))
# Serialize the compute graph to a file
torchforce_savedir = 'models/muller_2d_torchforce.pt'
module.save(torchforce_savedir)

#### 2. Umbrella Sampling on FMRC

In [ ]:
### Umbrella Sampling: create windows
# Harmonic potential as in the conventional umbrella sampling: V(x) = k*(x-x0)^2
# Spring constant
k_spring = 100
# Window centers
n_windows = 50
x_min = -3
x_max = 1.25
window_centers = np.linspace(x_min,x_max,n_windows)
# Select the configuration that is closest to the window center in the RC space
tree = cKDTree(r)
_, idx = tree.query(window_centers[:,None], k=1)  # nearest neighbor indices
center_configs = data_ref[idx]
print(center_configs.shape)

In [ ]:
def us_bias(x,x0,k):
    return 0.5*k*(x-x0)**2

def us_mbar_input(x,x0,k_spring,n_samples_per_window,unbiased=True,temp=310):
    kt = 8.314 * temp / 1000
    if unbiased == True:
        u_kn = np.zeros((len(x0)+1,len(x)))
        for k,x0_k in enumerate(x0):
            u_kn[k+1] = us_bias(x,x0_k,k_spring) / kt
        N_k = np.ones(n_windows+1,) * n_samples_per_window
        N_k[0] = 0
    else:
        u_kn = np.zeros((len(x0),len(x)))
        for k,x0_k in enumerate(x0):
            u_kn[k] = us_bias(x,x0_k,k_spring) / kt
        N_k = np.ones(n_windows,) * n_samples_per_window
    return u_kn,N_k

# Plot
plt.figure(figsize=(6,4))
plt.plot(bin_centers, neg_logpdf, marker='o', linestyle='-', color='black')
plt.xlabel("Value")
plt.ylabel("-log(pdf estimate)")
plt.title("Negative log of histogram PDF estimate")
plt.grid(True, linestyle="--", alpha=0.6)

window_range = (x_max-x_min) / n_windows + 0.15
x = np.linspace(-1,1,250)

for x0_i in window_centers:
    xw1 = np.linspace(x0_i-window_range,x0_i+window_range,100)
    bias_i = us_bias(xw1,x0_i,k_spring)
    plt.plot(xw1,bias_i)

plt.show()

In [ ]:
# Visualize
markersize = 3
stride = 1

fig,ax = plt.subplots()

sc = ax.scatter(data_ref[:,0][::stride],data_ref[:,1][::stride],c=r[::stride],s=markersize)
plt.colorbar(sc,label='RC')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim([MullerForce().x_range[0],MullerForce().x_range[1]])
ax.set_ylim([MullerForce().y_range[0],MullerForce().y_range[1]])

contour = ax.contour(X, Y, r_grid, levels=50, colors='black',linestyles='--')
ax.scatter(center_configs[:,0],center_configs[:,1],s=10,color='red')
#plt.colorbar(contour, label='RC')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
eq_steps = 250
n_iters_total = 100000
n_iters = int(n_iters_total / n_windows)
unbiased = True
torchforce_savedir = 'models/muller_2d_torchforce.pt'

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    # data in shape(no. of sims, no. of steps, 5), 5 for x, y, r, MBAR factor, girsanov factor
    data = np.zeros((len(window_centers),n_iters,5))
    
    save_dir = 'traj_and_dat/2d_muller/umbrella_sampling/{n_windows}x{n_iters}_k={k_spring}_umbrella_sampling_{n_re}.npy'.format(
        k_spring = k_spring, n_windows = n_windows, n_iters = n_iters, n_re = n_re
    )
    
    for i,center_config_i in enumerate(center_configs):
        print('Running the {i}/{n_windows} Window of Umbrella Sampling Simulation ...'.format(
            i = i, n_windows = n_windows
        ))
        # starting US simulation at current window center
        starting_position = np.array([[center_config_i[0],center_config_i[1],0]])
        ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
        ### using context.reinitialize()
        system = System()
        systemforce = MullerForce()
        system.addParticle(mass)
        systemforce.addParticle(0, [])
        system.addForce(systemforce)

        # Current window
        bias = CustomCVForce('0.5*{k}*(cv-{r0})^2'.format(k=k_spring,r0=window_centers[i]))
        cv = TorchForce(torchforce_savedir)
        bias.addCollectiveVariable('cv',cv)
        bias.setForceGroup(1)
        system.addForce(bias)

        topology = app.Topology()
        
        # Langevin integrator that allows girsanov reweighting factor output
        integrator = LangevinSplittingGirsanov(
            nstxout = nstxout, temperature = temperature,
            collision_rate = collision_rate, timestep = timestep,
            splitting = splitting
        )

        # Create simulation
        simulation = Simulation(topology,system,integrator,platform)
        simulation.context.setPositions(starting_position)
        simulation.context.setVelocitiesToTemperature(temperature)

        # Equilibrate inside the window
        simulation.step(eq_steps)
        
        for k in range(n_iters):
            if k % 100 == 0:
                print(str(k)+'/'+str(n_iters))
            # Record Positions
            data[i,k,:2] = simulation.context.getState(
                getPositions=True
            ).getPositions(asNumpy=True).value_in_unit(nanometer)[0,:2]
            # Step simulation
            simulation.step(nstxout)
            # Record Girsanov weight
            data[i,k,4] = simulation.integrator.getGlobalVariableByName("M")
            
    r_n = fmrc.transform(tica.transform(data[:,:,:2]))
    data[:,:,2] = r_n.reshape(n_windows,-1)
    
    r_concat_n = np.concatenate(r_n)
    n_samples_per_window = data[:,:,:2].shape[1]
    u_kn, N_k = us_mbar_input(
        r_concat_n.reshape(-1,), window_centers, k_spring, n_samples_per_window, unbiased, temp
    )
    # Fit MBAR
    mbar = MBAR(u_kn, N_k)
    data[:,:,3] = mbar.W_nk[:,0].reshape(n_windows,-1)
    np.save(save_dir,data)

#### 3. Steered MD on FMRC

In [ ]:
# Spring constant
k_spring = 100

# Pulling from -0.8 to 0.8 with pulling_iters steps, 
# and then reverse the pulling direction
x_min = -3
x_max = 1.25
pulling_iters = 2500
v_pulling = (x_max - x_min) / pulling_iters              # cv value/iteration

In [ ]:
# starting the pulling simulation at one end
dist = (r.reshape(-1,) - x_min)**2
idx = np.where(dist == dist.min())[0][0]
starting_position = np.array([[data_ref[idx][0],data_ref[idx][1],0]])
starting_position
x0_update_sign = 1.0       # we first increase x0 from minx to maxx

# Pulling Simulation
n_repeats = 10
n_iters = 100000

for n_re in range(n_repeats):
    save_dir = 'traj_and_dat/2d_muller/steered_md/{n_iters}_v_pulling={v_pulling}_steered_md_{n_re}.npy'.format(
            v_pulling = v_pulling, n_iters = n_iters, n_re = n_re
        )

    # data in shape(no. of sims, no. of steps, 4), 4 for x, y, r, girsanov factor
    data = np.zeros((n_iters,4))

    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = MullerForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Initialize Pulling force
    cv = TorchForce(torchforce_savedir)
    pullingforce = CustomCVForce('0.5 * {k_spring} * (cv-x0)^2'.format(k_spring=k_spring))
    pullingforce.addCollectiveVariable('cv',cv)
    pullingforce.addGlobalParameter('x0', x_min)
    pullingforce.setForceGroup(1)
    system.addForce(pullingforce)
        
    # Create a dummy topology object
    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )
    
    # Create simulation
    simulation = Simulation(topology,system,integrator)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    for k in range(n_iters):
        x0_k = simulation.context.getParameter('x0')
        # Printing
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
            print('Current x0: {x0_k}'.format(x0_k=x0_k))
        # Update x0
        if x0_k >= x_max:
            x0_update_sign = -1.0    # reverse the sign so x0 starts to decrease
            print('x0 will decrease now.')
        elif x0_k <= x_min:
            x0_update_sign = 1.0     # reverse the sign so x0 starts to increase
            print('x0 will increase now.')
        else:
            None
        
        data[k,:2] = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(nanometer)[0,:2]
        simulation.step(nstxout)
        data[k,3] = simulation.integrator.getGlobalVariableByName("M")
        
        x0_new = x0_k + x0_update_sign * v_pulling
        simulation.context.setParameter('x0',x0_new)

    data[:,2] = fmrc.transform(tica.transform(data[:,:2])).reshape(-1,)
    np.save(save_dir,data)